In [3]:
import requests
import time
import os
import subprocess
from Bio import SeqIO
import re
import urllib.parse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


### Get Data

In [22]:
#Dictionary of protein names and uniprot queries
query_dict = {
    "Phospholipase A2(PLA2)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:phospholipase OR protein_name:pla2)",
    "Snake Venom Metalloproteinases(SVMP)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Metalloproteinase OR protein_name:SVMP)",
    "Disintegrins": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:disintegrin)",
    "Snake Venom Serine Proteases(SVSP)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Serine Protease OR protein_name:SVSP)",
    "Three-Finger Toxins(3FTX)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:three-finger OR protein_name:3ftx)",
    #"Cysteine-Rich Secretory Proteins(CRISP)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:cysteine-rich OR protein_name:CRISP)",
    #"Kunitz-Type Protease Inhibitors": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:kunitz)",
    #"L-Amino Acid Oxidases(LAAO)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:L-Amino Acid Oxidase OR protein_name:LAAO)",
    #"Nerve Growth Factor(NGF)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:nerve growth factor OR protein_name:NGF)",
    #"Vascular Endothelial Growth Factor(VEGF)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Vascular Endothelial Growth Factor OR protein_name:VEGF)",
    #"Bradykinin-Potentiating Peptides(BPP)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Bradykinin-Potentiating Peptide OR protein_name:BPP)",
    #"Natriuretic Peptides(NP)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Natriuretic Peptide OR protein_name:NP)",
    "C-Type Lectins or Lectin-Like Proteins": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:C-Type Lectin OR protein_name:Lectin-Like Protein OR protein_name:CTL)",
    #"5'-Nucleotidases": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:5'-Nucleotidase OR protein_name:5'NT)",
    #"Hyaluronidases": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Hyaluronidase)",
    #"Phosphodiesterases (PDE)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Phosphodiesterase OR protein_name:PDE)",
    #"Neurotrophins (other than NGF)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Neurotrophin)",
    #"Glutaminyl Cyclase(QC)": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Glutaminyl Cyclase OR protein_name:QC)",
    #"Beta-bungarotoxin": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Beta-bungarotoxin)",
    #"Ohanin": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Ohanin)",
    #"Vespryns": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Vespryn)",
    #"Waprins": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Waprin)",
    #"Sarafotoxins": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Sarafotoxin)",
    #"Taicatoxin": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Taicatoxin)",
    #"Convulxin": "(taxonomy_id:8570) AND (venom OR toxin) AND (protein_name:Convulxin)"
}

#### Collect data with Uniprot Rest API

In [23]:
base_url = "https://rest.uniprot.org/uniprotkb/search"
max_size = 500  # max results per page

for protein_name, query in query_dict.items():
    print(f"Fetching sequences for {protein_name}...")
    all_results = []
    cursor = None
    total_count = 0
    
    while True:
        params = {
            "query": query,
            "format": "fasta",
            "size": max_size,
        }
        if cursor:
            params["cursor"] = cursor
        
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"Failed to retrieve data for {protein_name}: {response.status_code}")
            break
        
        data = response.text
        if not data.strip():
            break
        
        all_results.append(data)
        
        # Get 'x-total-results' header to know total
        if total_count == 0:
            total_count = int(response.headers.get("x-total-results", "0"))
        
        # Check if next page cursor is present in Link header
        link_header = response.headers.get("Link", "")
        next_cursor = None
        for link_part in link_header.split(","):
            if 'rel="next"' in link_part:
                # Example link_part: <https://rest.uniprot.org/uniprotkb/search?query=...&cursor=xyz>; rel="next"
                url_start = link_part.find("<") + 1
                url_end = link_part.find(">", url_start)
                url = link_part[url_start:url_end]
                parsed = urllib.parse.urlparse(url)
                qs = urllib.parse.parse_qs(parsed.query)
                cursor_list = qs.get("cursor")
                if cursor_list:
                    next_cursor = cursor_list[0]
                break
                
        if not next_cursor:
            break  # no more pages
        cursor = next_cursor
        
        time.sleep(1)  # be kind to server
    
    if all_results:
        fasta_text = "\n".join(all_results)
        filename = os.path.join("../raw_data", f"{protein_name.replace(' ', '_')}.fasta")
        with open(filename, "w") as f:
            f.write(fasta_text)
        print(f"Saved {total_count} sequences for {protein_name} to {filename}")
    else:
        print(f"No sequences found for {protein_name}")


Fetching sequences for Phospholipase A2(PLA2)...
Saved 1735 sequences for Phospholipase A2(PLA2) to ../raw_data/Phospholipase_A2(PLA2).fasta
Fetching sequences for Snake Venom Metalloproteinases(SVMP)...
Saved 1378 sequences for Snake Venom Metalloproteinases(SVMP) to ../raw_data/Snake_Venom_Metalloproteinases(SVMP).fasta
Fetching sequences for Disintegrins...
Saved 599 sequences for Disintegrins to ../raw_data/Disintegrins.fasta
Fetching sequences for Snake Venom Serine Proteases(SVSP)...
Saved 1102 sequences for Snake Venom Serine Proteases(SVSP) to ../raw_data/Snake_Venom_Serine_Proteases(SVSP).fasta
Fetching sequences for Three-Finger Toxins(3FTX)...
Saved 668 sequences for Three-Finger Toxins(3FTX) to ../raw_data/Three-Finger_Toxins(3FTX).fasta
Fetching sequences for C-Type Lectins or Lectin-Like Proteins...
Saved 873 sequences for C-Type Lectins or Lectin-Like Proteins to ../raw_data/C-Type_Lectins_or_Lectin-Like_Proteins.fasta


#### Parse .fasta Files
Create metadata.csv and protein_sequences.csv

In [22]:
def parse_uniprot_fasta_header(header):
    parts = header.split(' ', 1)
    uniprot_part = parts[0]  # e.g. sp|W8EFS0|V5NTD_MACLB
    rest = parts[1] if len(parts) > 1 else ""

    db, accession, uniprot_id = (uniprot_part.split('|') + ["", ""])[:3]

    title = rest.split(" OS=")[0].strip() if " OS=" in rest else rest.strip()

    os_match = re.search(r'OS=([^=]+?) (OX=|PE=|SV=|$)', rest)
    ox_match = re.search(r'OX=(\d+)', rest)
    pe_match = re.search(r'PE=(\d+)', rest)
    
    organism = os_match.group(1).strip() if os_match else ""
    taxonomy_id = ox_match.group(1).strip() if ox_match else ""
    evidence_level = pe_match.group(1).strip() if pe_match else ""

    return {
        "database": db,
        "database_id": accession,
        "uniprot_id": uniprot_id,
        "title": title,
        "organism": organism,
        "taxonomy_id": taxonomy_id,
        "evidence_level": evidence_level,
    }


def fasta_to_df(in_directory, metadata_path, protein_sequences_path):
    '''Fetches all .fasta files in "directoy" and creates, metadata and protein_sequence csvs.
    They are saved as metadata_out and protein_sequences_out respectively'''

    metadata_all = []
    sequences_all = []
    
    for filename in os.listdir(in_directory):
        if filename.endswith(".fasta") and not filename.endswith(".fasta.clst"):
            print(f"Processing: {filename}")
            protein_name = os.path.splitext(filename)[0]
            filepath = os.path.join(in_directory, filename)
            
            for record in SeqIO.parse(filepath, "fasta"):
                meta = parse_uniprot_fasta_header(record.description)
                meta["protein"] = protein_name
                metadata_all.append(meta)
                
                sequences_all.append({
                    "uniprot_id": meta["uniprot_id"],
                    "protein_name": protein_name,
                    "protein_sequence": str(record.seq)
                })

    df_meta = pd.DataFrame(metadata_all)
    df_seq = pd.DataFrame(sequences_all)

    df_meta.to_csv(metadata_path, index=False)
    df_seq.to_csv(protein_sequences_path, index=False)


    print("Metadata saved to " + metadata_path)
    print("Protein sequences saved to " + protein_sequences_path)

    return


In [18]:
#make new csvs from fasta files
fasta_to_df("../raw_data/", 
            "../raw_data/metadata/metadata.csv", 
            "../raw_data/protein_sequences/protein_sequences.csv")

Processing: Snake_Venom_Serine_Proteases(SVSP).fasta
Processing: Snake_Venom_Metalloproteinases(SVMP).fasta
Processing: Phospholipase_A2(PLA2).fasta
Processing: C-Type_Lectins_or_Lectin-Like_Proteins.fasta
Processing: Three-Finger_Toxins(3FTX).fasta
Processing: Disintegrins.fasta
Metadata saved to ../raw_data/metadata/metadata.csv
Protein sequences saved to ../raw_data/protein_sequences/protein_sequences.csv


#### Inspect metadata.csv and protein_sequences.csv

In [5]:
metadata_df = pd.read_csv("../raw_data/metadata/metadata.csv")
metadata_df.head()

,database,database_id,uniprot_id,title,organism,taxonomy_id,evidence_level,protein
0,sp,Q91516,VSPPA_TRIST,Venom plasminogen activator TSV-PA,Trimeresurus stejnegeri,39682,1,Snake_Venom_Serine_Proteases(SVSP)
1,sp,E0Y419,VSPBF_MACLB,Beta-fibrinogenase,Macrovipera lebetinus,3148341,1,Snake_Venom_Serine_Proteases(SVSP)
2,sp,Q9PTU8,VSP3_BOTJA,Snake venom serine protease BPA,Bothrops jararaca,8724,1,Snake_Venom_Serine_Proteases(SVSP)
3,sp,Q8AY79,VSPS2_TRIST,Beta-fibrinogenase stejnefibrase-2,Trimeresurus stejnegeri,39682,1,Snake_Venom_Serine_Proteases(SVSP)
4,sp,Q8JH85,VSPA_MACLB,Alpha-fibrinogenase,Macrovipera lebetinus,3148341,1,Snake_Venom_Serine_Proteases(SVSP)


In [6]:
sequences_df = pd.read_csv("../raw_data/protein_sequences/protein_sequences.csv")
sequences_df.head()

,uniprot_id,protein_name,protein_sequence
0,VSPPA_TRIST,Snake_Venom_Serine_Proteases(SVSP),MELIRVLANLLILQLSYAQKSSELVFGGDECNINEHRSLVVLFNSN...
1,VSPBF_MACLB,Snake_Venom_Serine_Proteases(SVSP),MVLIRVLANLLLLQLSHAQKSSELVVGGDECNINEHRSLVFLYNSS...
2,VSP3_BOTJA,Snake_Venom_Serine_Proteases(SVSP),MVLIRVIANLLILQLSNAQKSSELVIGGDECNITEHRFLVEIFNSS...
3,VSPS2_TRIST,Snake_Venom_Serine_Proteases(SVSP),MELIRVLANLLILQLSYAQKSSELVVGGDECNINEHRSLVAIFNST...
4,VSPA_MACLB,Snake_Venom_Serine_Proteases(SVSP),MVLIRVLANLVMLHLSYGEKSSELVIGGRPCNINQHRSLALLYNSS...


In [7]:
#Compare their shapes
print(metadata_df.shape)
print(sequences_df.shape)

(6355, 8)
(6355, 3)


In [8]:
metadata_df.groupby('protein').count()

,database,database_id,uniprot_id,title,organism,taxonomy_id,evidence_level
protein,,,,,,,
C-Type_Lectins_or_Lectin-Like_Proteins,873,873,873,873,873,873,873
Disintegrins,599,599,599,599,599,599,599
Phospholipase_A2(PLA2),1735,1735,1735,1735,1735,1735,1735
Snake_Venom_Metalloproteinases(SVMP),1378,1378,1378,1378,1378,1378,1378
Snake_Venom_Serine_Proteases(SVSP),1102,1102,1102,1102,1102,1102,1102
Three-Finger_Toxins(3FTX),668,668,668,668,668,668,668


In [9]:
sequences_df.groupby('protein_name').count()

,uniprot_id,protein_sequence
protein_name,,
C-Type_Lectins_or_Lectin-Like_Proteins,873,873
Disintegrins,599,599
Phospholipase_A2(PLA2),1735,1735
Snake_Venom_Metalloproteinases(SVMP),1378,1378
Snake_Venom_Serine_Proteases(SVSP),1102,1102
Three-Finger_Toxins(3FTX),668,668


### Make train/val/test folds

In [10]:
def make_train_val_test_splits(metadata):

    '''Takes the metadata file and uses it to create the train, validation, and test splits
    These will be saved in the csv in column "split" '''

    metadata_df = pd.read_csv(metadata)

    # Split into train/val/test (e.g., 80/10/10), strtified along species label since data is limited
    train_df, test_df = train_test_split(metadata_df, test_size=0.2, random_state=1992, stratify=metadata_df['protein'])
    test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=1992, stratify=test_df['protein'])

    # Add split column
    metadata_df["fold"] = None
    metadata_df.loc[train_df.index, "fold"] = "train"
    metadata_df.loc[val_df.index, "fold"] = "val"
    metadata_df.loc[test_df.index, "fold"] = "test"

    print(f'Shape of train fold: {train_df.shape}')
    print(f'Shape of val fold: {val_df.shape}')
    print(f'Shape of test fold: {test_df.shape}')

    metadata_df.to_csv(metadata)

    print('splits created and metadata updated at ' + metadata)
    return

In [11]:
#create train/val/test split for full dataset
make_train_val_test_splits("../raw_data/metadata/metadata.csv")

Shape of train fold: (5084, 8)
Shape of val fold: (636, 8)
Shape of test fold: (635, 8)
splits created and metadata updated at ../raw_data/metadata/metadata.csv


In [12]:
#lookup dictionary for converting labels
path = '../raw_data/metadata/metadata.csv'
metadata_df = pd.read_csv(path)
label_to_index = {label: index for index, label in enumerate(sorted(metadata_df['protein'].unique()))}
metadata_df['label_index'] = metadata_df['protein'].map(label_to_index)
metadata_df.to_csv(path)


In [13]:
print(label_to_index)

{'C-Type_Lectins_or_Lectin-Like_Proteins': 0, 'Disintegrins': 1, 'Phospholipase_A2(PLA2)': 2, 'Snake_Venom_Metalloproteinases(SVMP)': 3, 'Snake_Venom_Serine_Proteases(SVSP)': 4, 'Three-Finger_Toxins(3FTX)': 5}


#### Use CD-HIT to get rid of duplicate sequences

In [14]:
def run_cd_hit(input_fasta, output_fasta, identity=0.8):
    subprocess.run([
        "cd-hit",
        "-i", input_fasta,
        "-o", output_fasta,
        "-c", str(identity),
        "-n", "5"  # Word size required for this identity level
    ])


directory = "../raw_data" #directory to find .fasta files
output_dir = '../raw_data/Clustered_Fasta' #where the clustered .fasta files will be stored
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(directory):
    if filename.endswith(".fasta"):
        input = f"../raw_data/" + filename
        output = f"../raw_data/Clustered_Fasta/Clustered_" + filename

        run_cd_hit(input, output)

Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i
         ../raw_data/Snake_Venom_Serine_Proteases(SVSP).fasta
         -o
         ../raw_data/Clustered_Fasta/Clustered_Snake_Venom_Serine_Proteases(SVSP).fasta
         -c 0.8 -n 5

Started: Wed Jul 16 17:56:25 2025
                            Output                              
----------------------------------------------------------------
total seq: 1099
longest and shortest : 1715 and 11
Total letters: 301394
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 10M = 10M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 76M

Table limit with the given memory limit:
Max number of representatives: 1983211
Max number of word counting entries: 90413381

comparing sequences from          0  to       1099
.
     1099  finished        291  clusters

Approximated maximum memory consumption: 77M
writing new database
writing clusterin

In [23]:
#make new csvs for clustered data
fasta_to_df("../raw_data/Clustered_Fasta/", 
            "../raw_data/metadata/clustered_metadata.csv", 
            "../raw_data/protein_sequences/clustered_protein_sequences.csv")

Processing: Clustered_Disintegrins.fasta
Processing: Clustered_Snake_Venom_Serine_Proteases(SVSP).fasta
Processing: Clustered_Phospholipase_A2(PLA2).fasta
Processing: Clustered_Snake_Venom_Metalloproteinases(SVMP).fasta
Processing: Clustered_Three-Finger_Toxins(3FTX).fasta
Processing: Clustered_C-Type_Lectins_or_Lectin-Like_Proteins.fasta
Metadata saved to ../raw_data/metadata/clustered_metadata.csv
Protein sequences saved to ../raw_data/protein_sequences/clustered_protein_sequences.csv


In [24]:
#create train/val/test split for the clustered data
make_train_val_test_splits("../raw_data/metadata/clustered_metadata.csv")

Shape of train fold: (1547, 8)
Shape of val fold: (194, 8)
Shape of test fold: (193, 8)
splits created and metadata updated at ../raw_data/metadata/clustered_metadata.csv


In [ ]:
#lookup dictionary for converting labels
path = '../raw_data/metadata/clustered_metadata.csv'
metadata_df = pd.read_csv(path)
label_to_index = {label: index for index, label in enumerate(sorted(metadata_df['protein'].unique()))}
metadata_df['label_index'] = metadata_df['protein'].map(label_to_index)
metadata_df.to_csv(path)

In [25]:
#see new class distributions
path = '../raw_data/metadata/clustered_metadata.csv'
metadata_df = pd.read_csv(path)
metadata_df.groupby('protein').count()


,Unnamed: 0,database,database_id,uniprot_id,title,organism,taxonomy_id,evidence_level,fold
protein,,,,,,,,,
Clustered_C-Type_Lectins_or_Lectin-Like_Proteins,356,356,356,356,356,356,356,356,356
Clustered_Disintegrins,218,218,218,218,218,218,218,218,218
Clustered_Phospholipase_A2(PLA2),465,465,465,465,465,465,465,465,465
Clustered_Snake_Venom_Metalloproteinases(SVMP),372,372,372,372,372,372,372,372,372
Clustered_Snake_Venom_Serine_Proteases(SVSP),291,291,291,291,291,291,291,291,291
Clustered_Three-Finger_Toxins(3FTX),232,232,232,232,232,232,232,232,232


#### Create fragmented sequences of test data to measure robustness

In [26]:
def fragment_sequence_fixed(seq, frag_len):
    """
    Fragments a sequence into multiple overlapping fixed-length windows.
    If the sequence is shorter than frag_len, returns an empty list.
    """
    seq_len = len(seq)
    if seq_len < frag_len:
        return [seq]

    fragments = []
    stride = frag_len // 2
    for start in range(0, seq_len - frag_len + 1, stride):
        fragment = seq[start:start + frag_len]
        fragments.append(fragment)

    #keep tail fragment as well
    #if (seq_len - frag_len) % stride != 0:
    #    fragments.append(seq[-frag_len:])


    return fragments

In [27]:
def make_fixed_length_fragment_csv(sequences_df, frag_len):

    #load the sequences
    #sequences_df = pd.read_csv(sequences_path)

    #collect fragment data
    records = []

    for idx, row in sequences_df.iterrows():
        uniprot_id = row['uniprot_id']
        protein = row['protein_name']
        sequence = row['protein_sequence']
        
        frags = fragment_sequence_fixed(sequence, frag_len)

        for idx, frag in enumerate(frags):
            frag_id = f'{uniprot_id}_frag{idx}'
            records.append({'id': frag_id,
                            'protein': protein,
                            'protein_sequence': frag
                            })

    frag_df = pd.DataFrame(records)

    return frag_df

In [28]:
output_dir = '../raw_data/protein_sequences/fragmented_test_sequences'
os.makedirs(output_dir, exist_ok=True)

metadata_path = '../raw_data/metadata/clustered_metadata.csv'
metadata_df = pd.read_csv(metadata_path)

sequences_path = '../raw_data/protein_sequences/clustered_protein_sequences.csv'
sequences_df = pd.read_csv(sequences_path)

#reduce to test set only
test_indices = metadata_df[metadata_df['fold'] == 'test'].index.to_list()
sequences_df = sequences_df.iloc[test_indices]

frag_lengths = [200,150,100,75,50,25]
for frag_len in frag_lengths:
    frag_df = make_fixed_length_fragment_csv(sequences_df, frag_len)

    #make copmatible with dataloader later in pipeline
    frag_df['fold'] = 'test'
    label_to_index = {label: index for index, label in enumerate(sorted(frag_df['protein'].unique()))}
    frag_df['label_index'] = frag_df['protein'].map(label_to_index)
    #save the dataframe
    frag_df.to_csv(output_dir + f'/fragments_len{frag_len}.csv')

#### Turn k-mer features into .npy (!!!k-mer features need to already be created)

In [29]:

features_df = pd.read_csv('../raw_data/protein_sequences/processed_protein_features.csv')
features_df = features_df.drop(['uniprot_id', 'protein_name'], axis=1)
features_df.head()

,freq_A,freq_C,freq_D,freq_E,freq_F,freq_G,freq_H,freq_I,freq_K,freq_L,...,freq_YYM,freq_YYN,freq_YYP,freq_YYQ,freq_YYR,freq_YYS,freq_YYT,freq_YYV,freq_YYW,freq_YYY
0,0.064990,0.046122,0.060797,0.088050,0.027254,0.058700,0.035639,0.050314,0.056604,0.079665,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.064182,0.041408,0.070393,0.074534,0.026915,0.060041,0.033126,0.045549,0.043478,0.091097,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.049100,0.062193,0.055646,0.083470,0.031097,0.063830,0.027823,0.060556,0.083470,0.060556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.000000,0.029412,0.029412,0.029412,0.058824,0.000000,0.000000,0.088235,0.058824,0.088235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.053254,0.112426,0.071006,0.082840,0.011834,0.076923,0.005917,0.029586,0.053254,0.041420,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005988,0.0,0.0


In [31]:
embeddings_np = features_df.values.astype(np.float32) 
np.save("../processed_data/embeddings/kmer_embeddings.npy", embeddings_np)

